# Hangman — submission

Inference only: loads the trained checkpoint, plays all 250,000 words in
`test.txt` with an honest simulation (the policy sees the masked board and
nothing else), and writes `submission.csv`.

Runs in minutes and needs no internet. Set `TRAIN_FROM_SCRATCH = True` to
reproduce the weights from `train.txt` inside this notebook instead.


In [ ]:
TRAIN_FROM_SCRATCH = False   # audit switch: retrains everything from train.txt

import sys, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "/kaggle/input/hangman-src/src")
sys.path.insert(0, "/kaggle/working/src")

import torch
from hangman.data import load_words, find_competition_dir
from hangman.model import HangmanNet, ModelConfig
from hangman.policies import NeuralPolicy
from hangman.evaluate import evaluate, print_report
from hangman.submit import write_submission, validate_submission

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMP = find_competition_dir()
print("data:", COMP)
test_words = load_words(f"{COMP}/test.txt")
print(len(test_words), "test words")

In [ ]:
if TRAIN_FROM_SCRATCH:
    raise SystemExit("run notebooks/01_train.ipynb -- it is the reproducible training path")

ckpt = torch.load("/kaggle/input/hangman-weights/hangman_r2.pt", map_location=DEVICE)
cfg = ModelConfig(**ckpt["cfg"])
model = HangmanNet(cfg)
model.load_state_dict(ckpt["model"])

fusion = json.load(open("/kaggle/input/hangman-weights/inference.json"))["fusion"]
policy = NeuralPolicy(model, DEVICE, fusion=fusion)
print("loaded; fusion =", fusion)

In [ ]:
metrics = evaluate(test_words, policy, max_len=cfg.max_len)
print_report(metrics)

In [ ]:
write_submission(metrics["guesses"], "submission.csv")
validate_submission("submission.csv", expected_rows=len(test_words))